# Atividade Prática: Explorando Dados na Web com Python

In [1]:
import requests
import pandas as pd
import io
from bs4 import BeautifulSoup

## Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)

Prática de construção de requisições GET simples, envio de parâmetros e manipulação de cabeçalhos.

### Exercício 1.1: Requisição GET simples

Requisição GET para a API pública do JSONPlaceholder.

In [2]:
url = 'https://jsonplaceholder.typicode.com/posts'
resposta = requests.get(url, timeout=10)

print(resposta.status_code)
print(resposta.url)

200
https://jsonplaceholder.typicode.com/posts


### Exercício 1.2: Filtro com `params`

Uso do argumento `params` para filtrar os resultados por `userId` igual a 2.

In [3]:
parametros = {'userId': 2}
resposta = requests.get(url, params=parametros, timeout=10)

print(resposta.url)
print(len(resposta.json()))

https://jsonplaceholder.typicode.com/posts?userId=2
10


### Exercício 1.3: Cabeçalhos personalizados

Envio de um dicionário de `headers` com um `User-Agent` personalizado.

In [4]:
headers = {'User-Agent': 'ProjetoDados-Web/1.0'}
resposta = requests.get(url, headers=headers, timeout=10)

print(resposta.status_code)

200


### Exercício 1.4: Status HTTP e URL final

Impressão do `status_code` (esperado 200) e da URL montada pelo `requests`, sempre usando `timeout`.

In [5]:
resposta = requests.get(url, headers=headers, params={'userId': 2}, timeout=10)

print('Status:', resposta.status_code)
print('URL final:', resposta.url)

Status: 200
URL final: https://jsonplaceholder.typicode.com/posts?userId=2


## Nível 2: Intermediário (JSON, Erros e Arquivos Binários)

Manipulação de formatos de dados reais, criação de códigos robustos contra falhas e download de mídias.

### Exercício 2.1: Consulta de CEPs no ViaCEP

Loop sobre três CEPs, conversão das respostas com `.json()` e armazenamento em um DataFrame do Pandas.

In [6]:
ceps = ['01001000', '20040020', '30130010']
resultados = []

for cep in ceps:
    url = f'https://viacep.com.br/ws/{cep}/json/'
    resposta = requests.get(url, timeout=10)
    resultados.append(resposta.json())

df_ceps = pd.DataFrame(resultados)
df_ceps

,cep,logradouro,complemento,unidade,bairro,localidade,uf,estado,regiao,ibge,gia,ddd,siafi
0,01001-000,Praça da Sé,lado ímpar,,Sé,São Paulo,SP,São Paulo,Sudeste,3550308,1004,11,7107
1,20040-020,Praça Pio X,lado ímpar,,Centro,Rio de Janeiro,RJ,Rio de Janeiro,Sudeste,3304557,,21,6001
2,30130-010,Praça Sete de Setembro,,,Centro,Belo Horizonte,MG,Minas Gerais,Sudeste,3106200,,31,4123


### Exercício 2.2: Download seguro com `try/except`

Função de download que utiliza `raise_for_status()` para capturar erros HTTP e imprime mensagem amigável em caso de falha.

In [7]:
def download_seguro(url):
    try:
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()
        return resposta
    except requests.exceptions.HTTPError as erro:
        print(f'Erro HTTP: {erro}')
    except requests.exceptions.RequestException as erro:
        print(f'Erro na requisição: {erro}')
    return None

### Exercício 2.3: Download de imagem binária

Requisição para o Picsum Photos para baixar uma imagem aleatória. O conteúdo bruto (`resposta.content`) é salvo em um arquivo `.jpg` no modo `'wb'`.

In [8]:
resposta = requests.get('https://picsum.photos/400/400', timeout=10)

with open('imagem.jpg', 'wb') as arquivo:
    arquivo.write(resposta.content)

print('Imagem salva com sucesso.')

Imagem salva com sucesso.


## Nível 3: Avançado (Webscraping e Ética)

Extração de dados de páginas HTML onde não há APIs estruturadas, respeitando as boas práticas.

### Exercício 3.1: Verificação do `robots.txt`

Requisição ao arquivo `robots.txt` do site para verificar permissões de acesso antes da raspagem.

In [9]:
url_robots = 'https://books.toscrape.com/robots.txt'
resposta = requests.get(url_robots, timeout=10)

print(resposta.status_code)
print(resposta.text[:200])

404
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



### Exercício 3.2: Extração de títulos e preços

Uso do BeautifulSoup com `'html.parser'` para localizar elementos da página do Books to Scrape. Extração do título e do preço dos 5 primeiros livros.

In [10]:
url = 'https://books.toscrape.com/'
resposta = requests.get(url, timeout=10)
sopa = BeautifulSoup(resposta.text, 'html.parser')

livros = sopa.select('article.product_pod')[:5]

for livro in livros:
    titulo = livro.h3.a['title']
    preco = livro.select_one('.price_color').text
    print(f'{titulo} — {preco}')

A Light in the Attic — Â£51.77
Tipping the Velvet — Â£53.74
Soumission — Â£50.10
Sharp Objects — Â£47.82
Sapiens: A Brief History of Humankind — Â£54.23


### Exercício 3.3: Salvamento em CSV

Salvamento dos dados extraídos dos livros em um arquivo CSV com o Pandas.

In [11]:
dados = []
for livro in livros:
    dados.append({
        'titulo': livro.h3.a['title'],
        'preco': livro.select_one('.price_color').text
    })

pd.DataFrame(dados).to_csv('livros.csv', index=False)
print('Arquivo livros.csv salvo.')

Arquivo livros.csv salvo.


### Exercício 3.4: Tabela da Wikipedia com `read_html()`

Uso de `pandas.read_html()` combinado com `io.StringIO()` para capturar uma tabela da Wikipedia diretamente em um DataFrame.

In [15]:
url = 'https://pt.wikipedia.org/wiki/Lista_de_países_por_população'
resposta = requests.get(url, timeout=10)

tabelas = pd.read_html(io.StringIO(resposta.text))
df_wiki = tabelas[0]
df_wiki.head()

ValueError: No tables found